In [2]:
import requests, time
import pandas as pd

def get_results(driver_id, first_year, last_year=2026):
    rows = []
    for year in range(first_year, last_year + 1):
        url = f"https://api.jolpi.ca/ergast/f1/{year}/drivers/{driver_id}/results.json?limit=100"
        data = requests.get(url, timeout=30).json()
        for race in data["MRData"]["RaceTable"]["Races"]:
            r = race["Results"][0]
            rows.append({
                "year": year,
                "round": int(race["round"]),
                "race": race["raceName"],
                "grid": int(r["grid"]),
                "finish": int(r["position"]),
                "status": r["status"],
            })
        time.sleep(1)
    return pd.DataFrame(rows).sort_values(["year", "round"]).reset_index(drop=True)

drivers = {
    "verstappen": {"id": "max_verstappen", "start": 2015},
    "hamilton":   {"id": "hamilton",       "start": 2007},
}

for name, info in drivers.items():
    df = get_results(info["id"], info["start"])
    df.to_csv(f"{name}.csv", index=False)
    print(name, df.shape)

verstappen (253, 6)
hamilton (400, 6)


In [3]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

features = ["grid", "form_3", "form_5", "season_avg"]

def add_features(df):
    df = df.copy()
    df["form_3"] = df["finish"].shift(1).rolling(3).mean()
    df["form_5"] = df["finish"].shift(1).rolling(5).mean()
    df["season_avg"] = (
        df.groupby("year")["finish"]
          .transform(lambda s: s.shift(1).expanding().mean())
    )
    return df

def evaluate(name):
    df = pd.read_csv(f"{name}.csv")
    data = add_features(df).dropna()
    train = data[data.year < 2025]
    test = data[data.year == 2025]

    print(f"--- {name} ---")
    print("Baseline MAE:", round(mean_absolute_error(test["finish"], test["form_5"]), 2))

    ridge = Ridge().fit(train[features], train["finish"])
    print("Ridge MAE:   ", round(mean_absolute_error(test["finish"], ridge.predict(test[features])), 2))

    gbr = GradientBoostingRegressor(random_state=0).fit(train[features], train["finish"])
    print("GBR MAE:     ", round(mean_absolute_error(test["finish"], gbr.predict(test[features])), 2))

for name in drivers:
    evaluate(name)

--- verstappen ---
Baseline MAE: 2.93
Ridge MAE:    2.73
GBR MAE:      3.6
--- hamilton ---
Baseline MAE: 3.79
Ridge MAE:    3.2
GBR MAE:      4.81


In [ ]:
def predict_next(name, grid):
    df = pd.read_csv(f"{name}.csv")
    data = add_features(df).dropna()
    model = Ridge().fit(data[features], data["finish"])

    row = pd.DataFrame([{
        "grid": grid,
        "form_3": df["finish"].tail(3).mean(),
        "form_5": df["finish"].tail(5).mean(),
        "season_avg": df[df.year == 2026]["finish"].mean(),
    }])
    return model.predict(row)[0]

# Fill these in after qualifying
vegas_grid = {
    "verstappen": 1,
    "hamilton": 5,
}

for name, grid in vegas_grid.items():
    print(f"{name}: predicted P{predict_next(name, grid):.1f}")